# Entrenamiento y selección del mejor modelo

In [ ]:
# Celda 1 — Setup (Colab)
from google.colab import drive, userdata
drive.mount('/content/drive')

token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/cuellojoaquinn/ThoraxNet.git /content/ThoraxNet
!cp /content/drive/MyDrive/dataset.zip /content/
!unzip -q /content/dataset.zip -d /content/dataset

import sys
sys.path.append('/content/ThoraxNet/dev')

In [ ]:
# Celda 2 — Imports
import pandas as pd
import matplotlib.pyplot as plt
from src.data import get_loaders
from src import run_experiments, EXPERIMENTS

In [ ]:
# Celda 3 — Loaders
DATASET_ROOT = "/content/dataset/data"
OUTPUT_DIR   = "/content/drive/MyDrive/output"

train_loader, val_loader, test_loader = get_loaders(
    DATASET_ROOT, batch_size=128, num_workers=4
)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}")

In [ ]:
# Celda 4 — Correr experimentos
results = run_experiments(EXPERIMENTS, train_loader, val_loader, output_dir=OUTPUT_DIR)

In [ ]:
# Celda 5 — Tabla comparativa
rows = []
for name, history in results.items():
    rows.append({
        "Experimento": name,
        "train_loss":  f"{history['train_loss'][-1]:.4f}",
        "val_loss":    f"{history['val_loss'][-1]:.4f}",
    })

df = pd.DataFrame(rows)
display(df)

In [ ]:
# Celda 6 — Curvas de loss por experimento
fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 4))

for ax, (name, history) in zip(axes, results.items()):
    epochs = range(1, len(history['train_loss']) + 1)
    ax.plot(epochs, history['train_loss'], label='train')
    ax.plot(epochs, history['val_loss'],   label='val')
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()

plt.tight_layout()
plt.savefig('loss_curves.png')
plt.show()